# Current Measurement Test

## Instantiation

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep
from scipy import signal
from scipy.signal import decimate
from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq
%matplotlib inline
contacts = {
    "P1" : 1,
    "P2" : 2,
}

# qdac2 = QSTL_QDac2(
#     name = "qdac2",
#     address = "ASRL5::INSTR",
#     ramp_rate = 1,
#     i_threshold = 2e-9,
#     v_limit = 1.0,
#     contacts = contacts
# )
daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)
t_acqu = 10 # acquisition time in second
decimation = 500
meas_seg = 1
total_meas_time = 1800
t = np.arange(start=0, stop=t_acqu, step=1/daq.max_sampling_rate)
station = Station()

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251117_NoiseData/current_noise.db")
exp = load_or_create_experiment("2D sweep", "current_noise_data")
meas = Measurement(exp=exp, station=station)

meas_time = Parameter(name = "meas_time", label = "Measurement Time", unit = "s")
meas_freq = Parameter(name = "meas_freq", label = "Measurement Frequency", unit = "Hz")
meas_rms = Parameter(name = "meas_rms", label = "Measurement RMS", unit = "pA")
trace_data = Parameter(name = "trace_data", label = "Measurement Data", unit = "pA")
fft_data = Parameter(name = "fft_data", label = "FFT Data", unit = "A/Hz^-1/2")
elapsed_time = Parameter(name = "elapsed_time", label = "Elapsed Time", unit = "s")
meas.register_parameter(meas_time)
meas.register_parameter(meas_freq)
meas.register_parameter(elapsed_time)
meas.register_parameter(fft_data, setpoints=(meas_freq, elapsed_time))
meas.register_parameter(trace_data, setpoints=(meas_time, elapsed_time))
meas.register_parameter(meas_rms, setpoints=(elapsed_time,))

## Measurement w/o DC Voltage

In [20]:
start_time = time()

with meas.run() as datasaver:
    while True:
        num_of_samples = int(daq.max_sampling_rate * t_acqu)
        voltage_i = daq.read(
            ch = "Dev2/ai1",
            num_of_samples = num_of_samples
        )
        voltage_i = np.array(voltage_i)
        current_i = daq.convert_volts_to_amps(voltage_i)

        i_desired = decimate(current_i, decimation, ftype="fir", zero_phase=True)
        t_desired = np.linspace(0, t_acqu, len(i_desired))

        rms_pA = 1e12 * np.sqrt(np.mean((i_desired - np.mean(i_desired))**2))
        print(f"RMS noise {rms_pA} pA")

        f, II_den = signal.periodogram(
            i_desired,
            fs = daq.max_sampling_rate/decimation,
            window = "flattop",
            scaling = "density",
            return_onesided = True
        )

        current_time = time() - start_time

        datasaver.add_result(
            (elapsed_time, [current_time] * len(f)),
            (meas_freq, f),
            (fft_data, np.sqrt(II_den))
        )
        datasaver.add_result(
            (elapsed_time, [current_time] * len(t_desired)),
            (meas_time, t_desired),
            (trace_data, 1e12 * i_desired)
        )
        datasaver.add_result(
            (elapsed_time, current_time),
            (meas_rms, rms_pA)
        )
        sleep((meas_seg - 1) * t_acqu)
        if current_time > total_meas_time:
            break

Starting experimental run with id: 12. 
RMS noise 5.739310940189405 pA
RMS noise 5.691807114128096 pA
RMS noise 5.640089739131748 pA
RMS noise 5.642383590998854 pA
RMS noise 7.561614004545469 pA
RMS noise 6.82232370252282 pA
RMS noise 5.594393147780639 pA
RMS noise 5.620226584167291 pA
RMS noise 5.635838819304753 pA
RMS noise 5.630891492607599 pA
RMS noise 5.617698737851216 pA
RMS noise 5.613531856988997 pA
RMS noise 7.523142521352443 pA
RMS noise 7.450193338950047 pA
RMS noise 5.8419311510307885 pA
RMS noise 5.672115737696933 pA
RMS noise 5.667254636693849 pA
RMS noise 7.5180159549591306 pA
RMS noise 7.847939523106903 pA
RMS noise 5.534689868376589 pA
RMS noise 7.885940129360366 pA
RMS noise 5.6769389962402395 pA
RMS noise 5.672641598835487 pA
RMS noise 5.667619683279229 pA
RMS noise 7.04533351142913 pA
RMS noise 7.414383892394564 pA
RMS noise 5.5746437680317324 pA
RMS noise 7.5577188203793835 pA
RMS noise 7.583249160692732 pA
RMS noise 5.913523061249518 pA
RMS noise 5.600594544429065

## Measurement w/ DC Voltage

In [14]:
import json

start_time = time()
bias_v = 0
qdac2.reset()
qdac2.free_all_triggers()
qdac2.ext3.delay_s(0)
qdac2.v_limit = 1.2
qdac2.ramp_channels(["P1"], [bias_v])

with meas.run() as datasaver:
    datasaver.dataset.add_metadata(
        tag = "Device_Info",
        metadata = json.dumps(
            {
                "device_res" : 1e6
            }
        )
    )
    while True:
        num_of_samples = int(daq.max_sampling_rate * t_acqu)
        voltage_i = daq.read(
            ch = "Dev2/ai1",
            num_of_samples = num_of_samples
        )
        voltage_i = np.array(voltage_i)
        current_i = daq.convert_volts_to_amps(voltage_i)

        i_desired = decimate(current_i, decimation, ftype="fir", zero_phase=True)
        t_desired = np.linspace(0, t_acqu, len(i_desired))

        rms_pA = 1e12 * np.sqrt(np.mean((i_desired - np.mean(i_desired))**2))
        print(f"RMS noise {rms_pA} pA")

        f, II_den = signal.periodogram(
            i_desired,
            fs = daq.max_sampling_rate/decimation,
            window = "flattop",
            scaling = "density",
            return_onesided = True
        )

        current_time = time() - start_time

        datasaver.add_result(
            (elapsed_time, [current_time] * len(f)),
            (meas_freq, f),
            (fft_data, np.sqrt(II_den))
        )
        datasaver.add_result(
            (elapsed_time, [current_time] * len(t_desired)),
            (meas_time, t_desired),
            (trace_data, 1e12 * i_desired)
        )
        datasaver.add_result(
            (elapsed_time, current_time),
            (meas_rms, rms_pA)
        )
        sleep((meas_seg - 1) * t_acqu)
        if current_time > total_meas_time:
            break

Starting experimental run with id: 9. 
RMS noise 147.88681724920792 pA
RMS noise 144.03855663236303 pA
RMS noise 143.4792050219263 pA
RMS noise 142.34935765124447 pA
RMS noise 138.47962332795993 pA
RMS noise 142.33019324240263 pA
RMS noise 144.5899284056729 pA
RMS noise 143.9900495716654 pA
RMS noise 144.07051096279216 pA
